# Phase 3 — Content-Based Movie Recommendation

## Objective

Build a Content-Based Movie Recommendation System using movie genre information.

### Techniques
- TF-IDF Vectorization
- Cosine Similarity
- Pandas
- Scikit-learn

The system recommends movies with similar genre profiles to a selected movie.


In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


In [2]:
movies = pd.read_csv("../data/movies.csv")

print("Dataset Shape:", movies.shape)
movies.head()


Dataset Shape: (10329, 3)


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [3]:
print("Columns:", movies.columns.tolist())

print("\nMissing Values:")
print(movies.isnull().sum())

print("\nDuplicate Rows:", movies.duplicated().sum())


Columns: ['movieId', 'title', 'genres']

Missing Values:
movieId    0
title      0
genres     0
dtype: int64

Duplicate Rows: 0


In [4]:
movies["title"] = movies["title"].fillna("Unknown Movie").astype(str)
movies["genres"] = movies["genres"].fillna("Unknown").astype(str)

movies["genre_text"] = movies["genres"].str.replace(
    "|", " ", regex=False
)

movies[["title", "genres", "genre_text"]].head()


,title,genres,genre_text
0,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,Adventure Animation Children Comedy Fantasy
1,Jumanji (1995),Adventure|Children|Fantasy,Adventure Children Fantasy
2,Grumpier Old Men (1995),Comedy|Romance,Comedy Romance
3,Waiting to Exhale (1995),Comedy|Drama|Romance,Comedy Drama Romance
4,Father of the Bride Part II (1995),Comedy,Comedy


## TF-IDF Vectorization

TF-IDF converts the genre text into numerical feature vectors so that movies can be compared mathematically.


In [5]:
tfidf = TfidfVectorizer(stop_words="english")

tfidf_matrix = tfidf.fit_transform(movies["genre_text"])

print("TF-IDF Matrix Shape:", tfidf_matrix.shape)


TF-IDF Matrix Shape: (10329, 23)


In [6]:
similarity_matrix = cosine_similarity(tfidf_matrix)

print("Similarity Matrix Shape:", similarity_matrix.shape)


Similarity Matrix Shape: (10329, 10329)


In [7]:
title_to_index = pd.Series(
    movies.index,
    index=movies["title"]
).drop_duplicates()

title_to_index.head()


title
Toy Story (1995)                      0
Jumanji (1995)                        1
Grumpier Old Men (1995)               2
Waiting to Exhale (1995)              3
Father of the Bride Part II (1995)    4
dtype: int64

In [8]:
def recommend_movies(movie_title, n=10):
    if movie_title not in title_to_index:
        return pd.DataFrame(
            columns=["movieId", "title", "genres", "similarity"]
        )

    idx = title_to_index[movie_title]

    similarity_scores = list(
        enumerate(similarity_matrix[idx])
    )

    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    similarity_scores = similarity_scores[1:n+1]

    movie_indices = [index for index, score in similarity_scores]
    scores = [score for index, score in similarity_scores]

    recommendations = movies.iloc[movie_indices][
        ["movieId", "title", "genres"]
    ].copy()

    recommendations["similarity"] = scores

    return recommendations.reset_index(drop=True)


In [9]:
recommend_movies("Toy Story (1995)", 10)


,movieId,title,genres,similarity
0,2294,Antz (1998),Adventure|Animation|Children|Comedy|Fantasy,1.0
1,3114,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,1.0
2,3754,"Adventures of Rocky and Bullwinkle, The (2000)",Adventure|Animation|Children|Comedy|Fantasy,1.0
3,4016,"Emperor's New Groove, The (2000)",Adventure|Animation|Children|Comedy|Fantasy,1.0
4,4886,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy,1.0
5,33463,DuckTales: The Movie - Treasure of the Lost La...,Adventure|Animation|Children|Comedy|Fantasy,1.0
6,45074,"Wild, The (2006)",Adventure|Animation|Children|Comedy|Fantasy,1.0
7,53121,Shrek the Third (2007),Adventure|Animation|Children|Comedy|Fantasy,1.0
8,65577,"Tale of Despereaux, The (2008)",Adventure|Animation|Children|Comedy|Fantasy,1.0
9,91355,Asterix and the Vikings (Astérix et les Viking...,Adventure|Animation|Children|Comedy|Fantasy,1.0


In [10]:
def search_movies(keyword, limit=20):
    results = movies[
        movies["title"].str.contains(
            keyword,
            case=False,
            na=False,
            regex=False
        )
    ]

    return results[
        ["movieId", "title", "genres"]
    ].head(limit)

search_movies("Batman")


,movieId,title,genres
129,153,Batman Forever (1995),Action|Adventure|Comedy|Crime
524,592,Batman (1989),Action|Crime|Thriller
1118,1377,Batman Returns (1992),Action|Crime
1253,1562,Batman & Robin (1997),Action|Adventure|Fantasy|Thriller
2569,3213,Batman: Mask of the Phantasm (1993),Animation|Children
6068,26152,Batman (1966),Action|Adventure|Comedy
6280,27064,Batman & Mr. Freeze: Subzero (1998),Action|Animation|Children|Crime
6288,27155,"Batman/Superman Movie, The (1998)",Action|Adventure|Animation|Children|Fantasy|Sc...
6299,27311,Batman Beyond: Return of the Joker (2000),Action|Animation|Crime|Sci-Fi|Thriller
6641,33794,Batman Begins (2005),Action|Crime|IMAX


## Interpretation

The Content-Based model represents each movie using its genre profile. Cosine similarity measures the similarity between these profiles. Higher similarity values indicate more similar genre combinations.

This method does not require user-rating history; it relies on movie metadata.


# Phase 3 Conclusion

The Content-Based Movie Recommendation System was successfully implemented using TF-IDF and cosine similarity.

The system can search movies and generate a ranked list of movies similar to a selected title based on genre information.
